In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import re
import string
import spacy as sp
from spacy.lang.en.stop_words import STOP_WORDS
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [ ]:
nlp = sp.load("en_core_web_sm")

In [ ]:
path_file = "/content/Spam_SMS.csv"
data = pd.read_csv(path_file)
data.head()

,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
data.drop_duplicates(inplace=True)
data["Class"] = data["Class"].map({"ham":0,"spam":1})
data.head()

,Class,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
unics, counts = np.unique(data['Class'], return_counts=True)
print(unics)
print(counts)

[0 1]
[4518  641]


In [ ]:
data['Class'].plot(kind='hist', bins=20, title='Class')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
for _ in range(10):
  print(data["Message"][np.random.randint(0,len(data)-1)])

K.k..how is your sister kids?
I can send you a pic if you like :)
U don't know how stubborn I am. I didn't even want to go to the hospital. I kept telling Mark I'm not a weak sucker. Hospitals are for weak suckers.
Lara said she can loan me  &lt;#&gt; .
Good night. Am going to sleep.
There're some people by mu, I'm at the table by lambda
Is fujitsu s series lifebook good?
URGENT! We are trying to contact U. Todays draw shows that you have won a £800 prize GUARANTEED. Call 09050001295 from land line. Claim A21. Valid 12hrs only
Please don't text me anymore. I have nothing else to say.
Cool. So how come you havent been wined and dined before?


In [ ]:
def clean_message(message):
    message = message.lower()
    message = re.sub(f'[{string.punctuation}]', '', message)
    message = re.sub(r'@w+', ' ', message)
    message = re.sub(r"[^a-zA-Z]", " ", message).strip()
    message = re.sub(r'\d+', ' ', message)
    message = re.sub(r'\s+', ' ', message)
    doc = nlp(message)
    words = [token.lemma_ for token in doc if token.lemma_ not in STOP_WORDS]
    return ' '.join(words)

In [ ]:
print("Original text")
example = data["Message"][2]
print(example)
print("\nProcessed text")
clean_message(example)

Original text
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

Processed text


'free entry wkly comp win fa cup final tkts st text fa receive entry questionstd txt ratetcs apply s'

In [ ]:
messages = data["Message"].apply(clean_message)
target = data["Class"].values
target = target.reshape(-1,1)
print("Done")

Done


In [ ]:
max_len = max(len(message.split()) for message in messages)
print(f'Maximum length of a description: {max_len}')

vectorizer = TextVectorization(output_mode='int', split='whitespace',
                               standardize='strip_punctuation',
                               max_tokens=10000,
                               output_sequence_length=max_len)

vectorizer.adapt(messages)
print(f'Vocabulary size: {vectorizer.vocabulary_size()}')

Maximum length of a description: 71
Vocabulary size: 7228


In [ ]:
def make_dataset(messages, target, split=0.8, batch=16, buffer=10000):
    dataset = tf.data.Dataset.from_tensor_slices((messages, target))
    dataset = dataset.shuffle(buffer)

    total_size = len(train_premise)
    split_size = int(total_size * split)

    train_ds = dataset.take(split_size)
    test_ds = dataset.skip(split_size)

    train_ds = train_ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.batch(batch).prefetch(tf.data.AUTOTUNE)

    return train_ds, test_ds

In [ ]:
messages_vectorized = vectorizer(messages)
train_ds, test_ds = make_dataset(messages_vectorized, target)
print(f'Train size: {len(train_ds)}')
print(f'Test size: {len(test_ds)}')

Train size: 258
Test size: 65


In [ ]:
def make_model(vocab_size, embedding_dim, rnn_units, lr=0.001):
  # Create the model
  model = tf.keras.Sequential([

    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(rnn_units,
                                                       return_sequences=True)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(rnn_units,
                                                       return_sequences=True)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(rnn_units)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
  ])
  optim = tf.keras.optimizers.Adam(learning_rate=lr)
  loss = tf.keras.losses.BinaryCrossentropy(from_logits=False)

  model.compile(loss=loss,
                optimizer=optim,
                metrics=['accuracy'])
  return model

In [ ]:
def mycallbacks(monitor='val_accuracy', patience=3, verbose=1,
              filepath='best_model.keras'):
    # Early stopping callback
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor=monitor,
                                                      patience=patience,
                                                      verbose=verbose)

    # Model checkpoint callback
    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath=filepath,
                                                          save_best_only=True,
                                                          monitor=monitor,
                                                          verbose=verbose)

    return [early_stopping, model_checkpoint]

In [ ]:
vocab_size = vectorizer.vocabulary_size()
embedding_dim = 128
rnn_units = 256
model = make_model(vocab_size, embedding_dim, rnn_units)

In [ ]:
history = model.fit(train_ds, epochs=10, validation_data=test_ds,
                    callbacks=mycallbacks())

Epoch 1/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 15s 38ms/step - accuracy: 0.9005 - loss: 0.2832 - val_accuracy: 0.8895 - val_loss: 0.4093
Epoch 2/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 0.9192 - loss: 0.2635 - val_accuracy: 0.9932 - val_loss: 0.0250
Epoch 3/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 0.9913 - loss: 0.0362 - val_accuracy: 0.9981 - val_loss: 0.0132
Epoch 4/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.9953 - loss: 0.0157 - val_accuracy: 0.9961 - val_loss: 0.0129
Epoch 5/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.9963 - loss: 0.0115 - val_accuracy: 0.9971 - val_loss: 0.0042
Epoch 6/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.9983 - loss: 0.0063 - val_accuracy: 0.9990 - val_loss: 0.0027
Epoch 7/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.9992 - loss: 0.0033 - val_accuracy: 0.9952 - val_loss: 0.0115
Epoch 8/10
258/258 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.9983 - loss: 0.0065 - val_a

In [ ]:
def plot_history(history):
    fig, axs = plt.subplots(1, 2, figsize=(15, 6))

    # Plotando a Loss
    axs[0].plot(history.history['loss'])
    axs[0].plot(history.history['val_loss'])
    axs[0].set_title('Loss')
    axs[0].set_xlabel('Epochs')
    axs[0].set_ylabel('Loss')
    axs[0].legend(['train', 'val'], loc='upper right')

    # Plotando a Acurácia
    axs[1].plot(history.history['accuracy'])
    axs[1].plot(history.history['val_accuracy'])
    axs[1].set_title('Accuracy')
    axs[1].set_xlabel('Epochs')
    axs[1].set_ylabel('Accuracy')
    axs[1].legend(['train', 'val'], loc='lower right')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_history(history)